In [1]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

# Notebook is in .../Group Sprint 1/notebooks
BASE = Path("..").resolve()
RAW_PATH = BASE / "data" / "raw" / "understat_shots.csv"
CLEAN_DIR = BASE / "data" / "cleaned"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Only "NA" counts as missing. "None" in last_action is a genuine Understat
# label, and this stops the result depending on the pandas version.
df_raw = pd.read_csv(RAW_PATH, keep_default_na=False, na_values=["NA"])
print(df_raw.shape)
df_raw.head()

(51010, 44)


,league,season_start,season,match_id,shot_id,date,home_team,away_team,final_home_goals,final_away_goals,...,is_open_play,is_set_piece,from_corner,is_free_kick,is_header,is_assisted,team_goals_before,opp_goals_before,score_diff,game_state
0,EPL,2021,2021/22,16376,425564,2021-08-13,Brentford,Arsenal,2,0,...,True,False,False,False,True,False,0,0,0,Level
1,EPL,2021,2021/22,16376,425565,2021-08-13,Brentford,Arsenal,2,0,...,True,False,False,False,False,True,0,0,0,Level
2,EPL,2021,2021/22,16376,425567,2021-08-13,Brentford,Arsenal,2,0,...,True,False,False,False,False,True,0,0,0,Level
3,EPL,2021,2021/22,16376,425568,2021-08-13,Brentford,Arsenal,2,0,...,True,False,False,False,False,True,1,0,1,Leading
4,EPL,2021,2021/22,16376,425569,2021-08-13,Brentford,Arsenal,2,0,...,True,False,False,False,False,True,1,0,1,Leading


In [2]:
df_raw.info()
summary = pd.DataFrame({
    "dtype": df_raw.dtypes,
    "n_missing": df_raw.isna().sum(),
    "n_unique": df_raw.nunique(),
})
display(summary)
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate shot_id:", df_raw["shot_id"].duplicated().sum())
display(df_raw[["x", "y", "minute", "xg", "angle_to_goal"]].describe())

for c in ["season", "result", "situation", "shot_type", "last_action"]:
    print(f"\n{c}\n", df_raw[c].value_counts(dropna=False))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51010 entries, 0 to 51009
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   league             51010 non-null  object 
 1   season_start       51010 non-null  int64  
 2   season             51010 non-null  object 
 3   match_id           51010 non-null  int64  
 4   shot_id            51010 non-null  int64  
 5   date               51010 non-null  object 
 6   home_team          51010 non-null  object 
 7   away_team          51010 non-null  object 
 8   final_home_goals   51010 non-null  int64  
 9   final_away_goals   51010 non-null  int64  
 10  is_home            51010 non-null  bool   
 11  team               51010 non-null  object 
 12  opponent           51010 non-null  object 
 13  player             51010 non-null  object 
 14  player_id          51010 non-null  int64  
 15  assisted_by        37525 non-null  object 
 16  minute             510

,dtype,n_missing,n_unique
league,object,0,1
season_start,int64,0,6
season,object,0,6
match_id,int64,0,1960
shot_id,int64,0,51010
date,object,0,603
home_team,object,0,29
away_team,object,0,29
final_home_goals,int64,0,9
final_away_goals,int64,0,8


Duplicate rows: 0
Duplicate shot_id: 0


,x,y,minute,xg,angle_to_goal
count,51010.000000,51010.000000,51010.000000,51010.000000,51010.000000
mean,102.610553,40.310427,49.028739,0.120998,0.460090
std,10.840631,9.667416,26.869821,0.167282,0.284535
min,0.480000,0.480000,0.000000,0.000000,0.002059
25%,96.360004,33.520001,27.000000,0.029453,0.270710
50%,104.519998,40.239999,49.000000,0.057851,0.363659
75%,109.680002,47.200000,72.000000,0.105501,0.565852
max,119.639996,79.600000,105.000000,0.979893,2.846825



season
 2023/24    10524
2024/25     9878
2021/22     9775
2022/23     9644
2025/26     9524
2026/27     1665
Name: season, dtype: int64

result
 MissedShots    17630
BlockedShot    15076
SavedShot      11579
Goal            5526
ShotOnPost       987
OwnGoal          212
Name: result, dtype: int64

situation
 OpenPlay          37360
FromCorner         8623
SetPiece           3049
DirectFreekick     1484
Penalty             494
Name: situation, dtype: int64

shot_type
 RightFoot        25443
LeftFoot         16494
Head              8820
OtherBodyPart      253
Name: shot_type, dtype: int64

last_action
 Pass               18817
Cross               6450
None                6254
Aerial              4308
Rebound             2569
Chipped             2345
TakeOn              2277
Standard            1978
BallRecovery        1350
BallTouch           1217
HeadPass            1163
Throughball         1137
Dispossessed         271
LayOff               214
CornerAwarded        119
Foul           

In [3]:
df = df_raw.copy()

# 1. Own goals are not shot attempts at goal (xg = 0, positioned at the scorer's
#    own end). Their effect on the score is already in team_goals_before /
#    opp_goals_before for later shots.
df = df[df["result"] != "OwnGoal"]

# 2. Penalties: fixed location, ~82% conversion, no penalty flag in the kept
#    features. Set to False to keep them.
DROP_PENALTIES = True
if DROP_PENALTIES:
    df = df[~df["is_penalty"]]

df = df.copy()  # avoids SettingWithCopy warnings

# 3. last_action: "None" is Understat's label for no recorded prior action.
#    fillna is a safeguard in case any true NaNs appear.
df["last_action"] = df["last_action"].fillna("None")

# 4. Types
df["is_home"] = df["is_home"].astype(int)
df["is_assisted"] = df["is_assisted"].astype(int)
for c in ["team", "season", "shot_type", "last_action", "result"]:
    df[c] = df[c].astype("string").str.strip()

# 5. Sanity checks
assert df["xg"].between(0, 1).all()
assert df["x"].between(0, 120).all() and df["y"].between(0, 80).all()
assert df["minute"].between(0, 130).all()
assert (df["score_diff"] == df["team_goals_before"] - df["opp_goals_before"]).all()

print(df.shape)

(50304, 44)


In [4]:
# CamelCase -> snake_case, e.g. RightFoot -> right_foot
def to_snake(s):
    return re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()

print(df["shot_type"].value_counts(dropna=False))
shot_types = sorted(df["shot_type"].dropna().unique())

shot_type_cols = []
for st in shot_types:
    col = to_snake(st)
    df[col] = (df["shot_type"] == st).astype(int)
    shot_type_cols.append(col)

print("\nNew columns:", shot_type_cols)

# Check: every shot is exactly one type
assert (df[shot_type_cols].sum(axis=1) == 1).all()
display(df[["shot_type"] + shot_type_cols].head())

RightFoot        25021
LeftFoot         16293
Head              8784
OtherBodyPart      206
Name: shot_type, dtype: Int64

New columns: ['head', 'left_foot', 'other_body_part', 'right_foot']


,shot_type,head,left_foot,other_body_part,right_foot
0,Head,1,0,0,0
1,RightFoot,0,0,0,1
2,RightFoot,0,0,0,1
3,RightFoot,0,0,0,1
4,RightFoot,0,0,0,1


In [5]:
MIN_OBS = 100

la_counts = df["last_action"].value_counts(dropna=False)
print(la_counts)

keep_vals = sorted(la_counts[la_counts >= MIN_OBS].index)
dropped_vals = sorted(la_counts[la_counts < MIN_OBS].index)
print(f"\nKept ({len(keep_vals)}):", keep_vals)
print(f"Below {MIN_OBS} ({len(dropped_vals)}):", dropped_vals)

last_action_cols = []
for val in keep_vals:
    col = "last_action_" + to_snake(val)
    df[col] = (df["last_action"] == val).astype(int)
    last_action_cols.append(col)

# Check: each shot has at most one flag; rare values have none
assert (df[last_action_cols].sum(axis=1) <= 1).all()
n_none = (df[last_action_cols].sum(axis=1) == 0).sum()
print(f"\nShots with all flags 0 (rare last_action values): {n_none}")
display(df[["last_action"] + last_action_cols].head())

Pass               18817
Cross               6450
None                6177
Aerial              4298
Rebound             2567
Chipped             2345
TakeOn              2273
Standard            1484
BallRecovery        1349
BallTouch           1200
HeadPass            1163
Throughball         1137
Dispossessed         267
LayOff               214
Tackle               110
Foul                 103
CornerAwarded         97
BlockedPass           86
Goal                  47
End                   31
Interception          21
OffsidePass           17
Card                  11
Clearance              8
OffsideProvoked        8
Challenge              7
SubstitutionOn         4
GoodSkill              4
KeeperPickup           3
FormationChange        2
ShieldBallOpp          1
ChanceMissed           1
Punch                  1
Start                  1
Name: last_action, dtype: Int64

Kept (16): ['Aerial', 'BallRecovery', 'BallTouch', 'Chipped', 'Cross', 'Dispossessed', 'Foul', 'HeadPass', 'LayOff', 

,last_action,last_action_aerial,last_action_ball_recovery,last_action_ball_touch,last_action_chipped,last_action_cross,last_action_dispossessed,last_action_foul,last_action_head_pass,last_action_lay_off,last_action_none,last_action_pass,last_action_rebound,last_action_standard,last_action_tackle,last_action_take_on,last_action_throughball
0,Aerial,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Throughball,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,BallRecovery,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Pass,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
4,Chipped,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
# Guard against running this cell twice
assert set(df["result"].unique()) <= {"Goal", "MissedShots", "BlockedShot", "SavedShot", "ShotOnPost"}, \
    "result already converted; rerun from the top"

print(df["result"].value_counts())
df["result"] = (df["result"] == "Goal").astype(int)

print("\nAfter conversion:")
print(df["result"].value_counts())
print("Goal rate:", df["result"].mean().round(4))

MissedShots    17617
BlockedShot    15076
SavedShot      11515
Goal            5119
ShotOnPost       977
Name: result, dtype: Int64

After conversion:
0    45185
1     5119
Name: result, dtype: int64
Goal rate: 0.1018


In [7]:
KEEP_COLS = [
    "is_home", "team", "season",
    "xg", "minute", "result",
    "x", "y", "angle_to_goal",
    "is_assisted",
    "team_goals_before", "opp_goals_before", "score_diff",
] + shot_type_cols + last_action_cols

df_clean = df[KEEP_COLS].reset_index(drop=True)

print("Dropped columns:", sorted(set(df_raw.columns) - set(KEEP_COLS)))
print(df_clean.shape)
print(df_clean.isna().sum().sum(), "missing values remaining")
assert set(df_clean["result"].unique()) == {0, 1}
display(df_clean.head())

out_path = CLEAN_DIR / "shots_cleaned.csv"
df_clean.to_csv(out_path, index=False)
print("Saved to", out_path)

Dropped columns: ['assisted_by', 'away_team', 'blocked', 'date', 'distance_m', 'final_away_goals', 'final_home_goals', 'from_corner', 'game_state', 'hit_post', 'home_team', 'is_free_kick', 'is_goal', 'is_header', 'is_open_play', 'is_own_goal', 'is_penalty', 'is_set_piece', 'last_action', 'league', 'match_id', 'on_target', 'opponent', 'player', 'player_id', 'season_start', 'shot_id', 'shot_type', 'situation', 'x_m', 'y_m']
(50304, 33)
0 missing values remaining


,is_home,team,season,xg,minute,result,x,y,angle_to_goal,is_assisted,...,last_action_foul,last_action_head_pass,last_action_lay_off,last_action_none,last_action_pass,last_action_rebound,last_action_standard,last_action_tackle,last_action_take_on,last_action_throughball
0,1,Brentford,2021/22,0.053464,10,0,109.560004,43.120001,0.714539,0,...,0,0,0,0,0,0,0,0,0,0
1,1,Brentford,2021/22,0.118297,11,0,108.960004,25.200000,0.288597,1,...,0,0,0,0,0,0,0,0,0,1
2,1,Brentford,2021/22,0.052145,21,1,104.880002,55.840002,0.275243,1,...,0,0,0,0,0,0,0,0,0,0
3,1,Brentford,2021/22,0.065706,27,0,97.439996,38.239999,0.364677,1,...,0,0,0,0,1,0,0,0,0,0
4,1,Brentford,2021/22,0.080691,29,0,107.039996,28.560001,0.376828,1,...,0,0,0,0,0,0,0,0,0,0


Saved to /home/jovyan/Data Science Toolbox/Group Sprint 1/data/cleaned/shots_cleaned.csv
